# Cardiogenesis: training data for RADT

Builds the RADT training dataset for the `Cardiogenesis` environment (Section 5, Results 3; Appendix C.2):

1. **Random-policy data.** From each of the 6 attractors of the Boolean network, take `STEPS_PER_ATTRACTOR` random actions. Each action flips one gene and is followed by `K` asynchronous network updates. The resulting random walks are split into trajectories of length `TRAJ_LEN`.
2. **Hindsight relabeling.** Goals are relabeled with the last state of each trajectory (done at training time). Each trajectory gets one avoid state, sampled from the `TOP_N` most visited states, and `z = 1` if the trajectory never visits it. A copy of every trajectory with no avoid state and `z = 1` is appended.

Avoid tokens are the 15-dimensional avoid states themselves.

Run from `raDT/radt` with the `radt` environment active. The dataset is written to `RADT_DATASET_DIR` (set in `raDT/constants.py`).

In [ ]:
STEPS_PER_ATTRACTOR = 10000   # random actions taken from each attractor (6 x 10000 = 60000 steps in total)
K = 10                        # asynchronous Boolean network updates per step
TRAJ_LEN = 30                 # trajectory length
TOP_N = 8                     # avoid states are sampled from the TOP_N most visited states
SEED = None
OUTPUT = 'cardiogenesis_relabeled.pkl'

In [ ]:
import os
import pickle
from copy import deepcopy

import numpy as np
import pandas as pd
from pyboolnet.file_exchange import bnet2primes
from pyboolnet.state_transition_graphs import random_walk
from pyboolnet.attractors import compute_attractors

from envs.cardiogenesis import GRN, to_array
from raDT.constants import RADT_DATASET_DIR

if SEED is not None:
    np.random.seed(SEED)

## Random-policy data

In [ ]:
bnet = bnet2primes(GRN)
attractors = [a["state"]['str'] for a in compute_attractors(bnet, "asynchronous")["attractors"]]
print(attractors)

def random_policy_walk(initial_state, length, k):
    states_readable, actions = [initial_state], []
    state = initial_state
    for _ in range(length):
        action = np.random.randint(len(initial_state))
        actions.append(action)
        perturbed = state[:action] + str(1 - int(state[action])) + state[action + 1:]
        path = random_walk(bnet, "asynchronous", initial_state=perturbed, length=k)
        state = "".join([str(x) for x in path[-1].values()])
        states_readable.append(state)
    return states_readable, actions

walks = [random_policy_walk(a, STEPS_PER_ATTRACTOR, K) for a in attractors]

In [ ]:
# split each random walk into trajectories of length TRAJ_LEN
trajectories = []
for states_readable, actions in walks:
    for j in range(len(states_readable) // TRAJ_LEN):
        s = states_readable[j * TRAJ_LEN : j * TRAJ_LEN + TRAJ_LEN + 1]
        a = actions[j * TRAJ_LEN : j * TRAJ_LEN + TRAJ_LEN]
        traj = {}
        traj["observations_r"] = np.array(s[:-1])
        traj["next_observations_r"] = np.array(s[1:])
        traj["observations"] = np.array([to_array(x) for x in s[:-1]])
        traj["next_observations"] = np.array([to_array(x) for x in s[1:]])
        traj["actions"] = np.array(a)
        traj['timesteps'] = traj["observations"].shape[0]
        traj['terminals'] = np.array([False] * (traj['timesteps'] - 1) + [True])
        traj['rewards'] = np.zeros(traj["actions"].shape)
        traj['costs'] = np.zeros(traj["actions"].shape)
        trajectories.append(traj)
print(len(trajectories), 'trajectories')

## Hindsight avoid-state relabeling

In [ ]:
visit_counts = pd.Series([s for states_readable, _ in walks for s in states_readable]).value_counts()
most_visited = list(visit_counts.index[:TOP_N])
print(visit_counts[:TOP_N])

copies = deepcopy(trajectories)
for traj in trajectories:
    avoid_state = most_visited[np.random.randint(TOP_N)]
    traj["success"] = 0 if avoid_state in traj["observations_r"] else 1
    traj["avoid_states"] = np.array([to_array(avoid_state)])

for traj in copies:
    traj["success"] = 1
    traj["avoid_states"] = np.array([])

print('fraction of successful avoid demonstrations (first pass):', np.mean([t["success"] for t in trajectories]))
dataset = trajectories + copies

In [ ]:
os.makedirs(RADT_DATASET_DIR, exist_ok=True)
with open(os.path.join(RADT_DATASET_DIR, OUTPUT), 'wb') as f:
    pickle.dump(dataset, f)
print('saved', os.path.join(RADT_DATASET_DIR, OUTPUT), len(dataset), 'trajectories')